# 🎓 Maestría en Inteligencia Artificial Aplicada
## Proyecto Integrador — Pipeline RAG con Max-Min Semantic Chunking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación Aumentada por Generación (RAG) utilizando el algoritmo de chunking semántico Max-Min sobre políticas corporativas en PDF.

**Flujo del notebook:**
1. Instalación de dependencias
2. Configuración del entorno y carga de modelos
3. Funciones de carga, limpieza y normalización
4. Algoritmo Max-Min Semantic Chunking
5. Pipeline principal de ejecución
6. Configuración del cliente Gemini
7. Generación del Golden Dataset con LLM
8. Motor de evaluación y reporte de métricas

In [1]:
# Instalación de dependencias del proyecto.
# Ejecutar solo una vez al iniciar el entorno de Colab.
!pip install -q sentence-transformers PyMuPDF tiktoken google-genai


In [2]:
# =============================================================
# SECCIÓN 1 — Imports, Configuración Global y Carga de Modelos
# =============================================================

import math
import re
import time
import uuid
from pathlib import Path
from typing import List

import fitz           # PyMuPDF — lectura de PDFs
import numpy as np
import pandas as pd
import tiktoken
from sentence_transformers import SentenceTransformer, util
from google.colab import drive

# ── Configuración de rutas ────────────────────────────────────────────────────
# Se intenta montar Google Drive; si falla, se trabaja en almacenamiento local.
try:
    drive.mount("/content/drive")
    DRIVE_PATH = Path(
        "/content/drive/MyDrive/Colab_Notebooks/MNA/Proyecto_Integrador-main"
    )
    if DRIVE_PATH.exists():
        PROJECT_ROOT = DRIVE_PATH
        print(f"✔  Ruta de Drive encontrada: {PROJECT_ROOT}")
    else:
        print("⚠  Ruta de Drive no encontrada. Se usará entorno local.")
        PROJECT_ROOT = Path("/content/proyecto_integrador")
except Exception as exc:
    print(f"⚠  No se pudo montar Drive ({exc}). Se usará entorno local.")
    PROJECT_ROOT = Path("/content/proyecto_integrador")

# Directorio donde deben estar los PDFs fuente
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)

print(f"\nDirectorio de datos: {RAW_DATA_PATH}")
print(
    "NOTA: Si los archivos no están ahí, "
    "súbelos mediante el panel lateral de Colab."
)

# ── Constantes de modelos ─────────────────────────────────────────────────────
TOKENIZER_MODEL = "cl100k_base"                # Tokenizador compatible con GPT-4
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# ── Inicialización de modelos ─────────────────────────────────────────────────
encoder = tiktoken.get_encoding(TOKENIZER_MODEL)

print(f"\nCargando modelo de embeddings: {EMBEDDING_MODEL_NAME} ...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("✔  Modelos cargados correctamente.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚠  Ruta de Drive no encontrada. Se usará entorno local.

Directorio de datos: /content/proyecto_integrador/data/raw
NOTA: Si los archivos no están ahí, súbelos mediante el panel lateral de Colab.

Cargando modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✔  Modelos cargados correctamente.


In [30]:
# =============================================================
# SECCIÓN 2 — Funciones de Carga y Limpieza de Documentos
# =============================================================


def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extrae todo el texto de un archivo PDF página por página.

    Parameters
    ----------
    pdf_path : str
        Ruta absoluta al archivo PDF.

    Returns
    -------
    str
        Texto concatenado de todas las páginas.
    """
    doc = fitz.open(pdf_path)
    pages_text = [page.get_text() for page in doc]
    return "\n".join(pages_text)


def load_documents(base_path: Path) -> pd.DataFrame:
    """
    Carga documentos PDF, TXT o MD desde un directorio raíz de forma recursiva.

    Parameters
    ----------
    base_path : Path
        Directorio raíz donde se buscan los archivos.

    Returns
    -------
    pd.DataFrame
        DataFrame con columnas: file, path, text.
        Devuelve un DataFrame vacío si no hay archivos válidos.
    """
    if not base_path.exists():
        print(f"Advertencia: La ruta {base_path} no existe. Creándola...")
        base_path.mkdir(parents=True, exist_ok=True)
        return pd.DataFrame(columns=["file", "path", "text"])

    docs = []
    for path in base_path.rglob("*"):
        try:
            suffix = path.suffix.lower()
            if suffix == ".pdf":
                text = extract_text_from_pdf(str(path))
            elif suffix in {".txt", ".md"}:
                text = path.read_text(encoding="utf-8", errors="ignore")
            else:
                continue  # Tipo de archivo no soportado

            if text.strip():  # Ignorar archivos vacíos
                docs.append({"file": path.name, "path": str(path), "text": text})

        except Exception as exc:
            print(f"Error al leer '{path.name}': {exc}")

    return pd.DataFrame(docs)


class NoiseReducer:
    """
    Limpia ruido frecuente en documentos corporativos.

    Elimina: encabezados repetitivos, pies de página legales,
    metadatos de control de versiones, marcas de agua y campos
    vacíos de formularios.
    """

    def __init__(self):
        self._patterns1 = [
            r"page\s+\d+(\s+of\s+\d+)?",
            r"página\s+\d+(\s+de\s+\d+)?",
            r"este documento.*confidencial.*",
            r"Spin\s+HOJA\s+\d+\s+de\s+\d+",
            r"Código\s+ND-TIFS-PEC-\d+",
            r"Dirección Administración y Finanzas",
            r"Versión:\s+\d+\.\d+",
            r"Fecha de creación:.*",
            r"Fecha de modificación.*",
            r"Estatus\s+Vigente",
        ]

        self._patterns2 = [
            r"page\s+\d+(\s+of\s+\d+)?",
            r"página\s+\d+(\s+de\s+\d+)?",
            r"pág\.\s*\d+",
            r"-\s*\d+\s*-",
            r"(todos los derechos reservados|all rights reserved).*",
            r"(confidencial|confidential|strictly confidential)",
            r"uso interno.*",
            r"para uso exclusivo de.*",
            r"este (documento|archivo) es propiedad de.*",
            r"versión:?\s*\d+[\.\d]*",
            r"rev\.?\s*\d+",
            r"fecha de (creación|emisión|vigencia|modificación|actualización):.*",
            r"elabor[oó] por:.*",
            r"revis[oó] por:.*",
            r"aprobó:.*",
            r"autorizado por:.*",
            r"estatus:?\s*(vigente|obsoleto|borrador)",
            r"folio\s*[:#]?\s*[\w\-]+",
            r"(clave|código|referencia|ref\.?)\s*[:#]?\s*[\w\-]+",
            r"expediente\s*[:#]?\s*[\w\-]+",
            r"no\.?\s*de\s*(documento|doc\.?)\s*[:#]?\s*[\w\-]+",
            r"(borrador|draft|muestra|sample|copia|copy)",
            r"no\s+válido\s+sin\s+firma.*",
            r"documento\s+controlado.*",
            r"_{3,}",
            r"\[[\s]*\]",
            r"N/?A",
            r":\s*\n",
            r"política\s+(interna|corporativa)\s*(de|para)?\s*[\w\s]+",
            r"procedimiento\s+[A-Z]{2,}-\d+",
            r"instructivo\s+[A-Z]{2,}-\d+",
            r"(aplica|aplicable)\s+a\s+(todo(s)?|el personal).*",
            r"alcance:.*",
            r"objetivo(s)?:.*",
            r"firma(s)?\s*(del?|de la)?\s*(responsable|director|gerente|autoridad).*",
            r"nombre\s+y\s+firma.*",
            r"cargo:?\s*\n",
            r"fecha\s+de\s+firma:.*",
            r"puesto:?\s*\n",
            r"véase\s+(también|además)?\s*(la\s+)?(política|sección|anexo|apartado).*",
            r"(de\s+acuerdo|conforme)\s+(a|con)\s+(la\s+)?(política|norma|lineamiento).*",
            r"según\s+(lo\s+establecido\s+en|la\s+política).*",
            r"queda\s+estrictamente\s+prohibido.*",
            r"el\s+incumplimiento\s+(de\s+esta\s+política)?.*sanción.*",
            r"cualquier\s+excepción\s+debe\s+ser\s+autorizada.*",
            r"la\s+empresa\s+se\s+reserva\s+el\s+derecho.*",
            r"sujeto\s+a\s+(cambios|modificaciones)\s+sin\s+previo\s+aviso.*",
            r"historial\s+de\s+(cambios|revisiones|modificaciones)",
            r"descripción\s+del\s+cambio.*",
            r"(revisión|revisado)\s+#?\d+.*",
            r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\s+.*\s+(aprobó|revisó|elaboró).*",
            r"distribución:.*",
            r"dirigido\s+a:.*",
            r"destinatarios:.*",
            r"lista\s+de\s+distribución.*",
        ]

        # Patrón activo por defecto
        self.patterns = self._patterns1

    def change_pattern(self, pattern_number: int):
        """Cambia el conjunto de patrones activo."""
        options = {
            1: self._patterns1,
            2: self._patterns2,
        }
        if pattern_number not in options:
            raise ValueError(f"Número inválido. Opciones: {list(options.keys())}")

        self.patterns = options[pattern_number]
        print(f"✅ Usando patterns{pattern_number} ({len(self.patterns)} patrones activos)")

    def clean(self, text: str) -> str:
        for pattern in self.patterns:
            text = re.sub(pattern, "", text, flags=re.IGNORECASE | re.MULTILINE)
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"\n\s*\n", "\n\n", text)
        return text.strip()


def normalize_text(text: str) -> str:
    """
    Normalización final: une líneas sueltas (líneas rotas) en párrafos
    y elimina espacios redundantes.

    Parameters
    ----------
    text : str
        Texto limpio.

    Returns
    -------
    str
        Texto normalizado, listo para el proceso de chunking.
    """
    # Dos o más saltos de línea → separador de párrafo
    text = re.sub(r"\n{2,}", "\n\n", text)
    # Salto de línea único (línea rota) → espacio
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    return text.strip()


def split_into_sentences(text: str) -> List[str]:
    """
    Divide el texto en oraciones usando puntuación terminal básica (.!?).

    Parameters
    ----------
    text : str
        Texto normalizado.

    Returns
    -------
    List[str]
        Lista de oraciones con al menos 6 caracteres.
    """
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 5]


In [31]:
# =============================================================
# SECCIÓN 3 — Algoritmo Max-Min Semantic Chunking
# Referencia: Kiss et al. (2025)
# =============================================================


def get_embeddings(sentences: List[str]):
    """
    Codifica una lista de oraciones en el espacio de embeddings.

    Parameters
    ----------
    sentences : List[str]
        Oraciones a codificar.

    Returns
    -------
    torch.Tensor
        Tensor de embeddings de forma (n, d).
    """
    return embedding_model.encode(sentences, convert_to_tensor=True)


def sigmoid(x: float) -> float:
    """
    Función sigmoide estándar, usada como factor de escala en el umbral
    dinámico del algoritmo Max-Min.

    Parameters
    ----------
    x : float
        Valor de entrada (en este contexto, el tamaño del chunk actual).

    Returns
    -------
    float
        Valor en el rango (0, 1).
    """
    return 1 / (1 + math.exp(-x))


def max_min_chunking(
    text: str,
    hard_thr: float = 0.6,
    c: float = 0.9,
    init_const: float = 1.5,
) -> List[str]:
    """
    Implementación del algoritmo Max-Min Semantic Chunking.

    Agrupa oraciones en chunks garantizando alta coherencia interna.
    La decisión de corte usa un umbral dinámico que combina:

    - **min_sim(C)**: similitud mínima dentro del chunk (cohesión interna).
    - **max_sim(sk, C)**: similitud máxima de la nueva oración con el chunk.
    - **hard_thr**: umbral duro como piso de seguridad.
    - **sigmoid**: escala el umbral según el tamaño del chunk.

    Parameters
    ----------
    text : str
        Texto normalizado a segmentar.
    hard_thr : float
        Umbral mínimo de similitud para añadir una oración al chunk actual.
    c : float
        Factor de escala del umbral dinámico (0 < c ≤ 1).
    init_const : float
        Multiplicador para la comparación inicial (chunk de una sola oración).

    Returns
    -------
    List[str]
        Lista de chunks de texto semánticamente coherentes.
    """
    sentences = split_into_sentences(text)
    if not sentences:
        return []

    embeddings = get_embeddings(sentences)
    chunks: List[str] = []
    current_indices: List[int] = []

    for k, embedding_k in enumerate(embeddings):

        # ── Inicio: primer chunk siempre arranca con la primera oración ──────
        if not current_indices:
            current_indices.append(k)
            continue

        # ── Chunk de una sola oración: comparación directa ───────────────────
        if len(current_indices) == 1:
            prev_idx = current_indices[0]
            sim = util.cos_sim(embeddings[prev_idx], embedding_k).item()

            if init_const * sim > hard_thr:
                current_indices.append(k)
            else:
                # Corte: la oración nueva es muy distinta → iniciar nuevo chunk
                chunks.append(" ".join(sentences[i] for i in current_indices))
                current_indices = [k]
            continue

        # ── Chunk con múltiples oraciones: umbral dinámico ───────────────────
        chunk_embeddings = embeddings[current_indices]

        # Similitud mínima interna del chunk (cohesión)
        cos_scores = util.cos_sim(chunk_embeddings, chunk_embeddings)
        mask = ~np.eye(len(chunk_embeddings), dtype=bool)
        min_sim_chunk = cos_scores[mask].min().item() if mask.any() else 1.0

        # Similitud máxima de la nueva oración respecto al chunk completo
        max_sim_new = util.cos_sim(embedding_k, chunk_embeddings).max().item()

        # Umbral dinámico: sube a medida que el chunk crece (vía sigmoide)
        dynamic_thr = max(
            c * min_sim_chunk * sigmoid(len(current_indices)),
            hard_thr,
        )

        if max_sim_new >= dynamic_thr:
            current_indices.append(k)
        else:
            # Corte: guardar chunk actual e iniciar uno nuevo
            chunks.append(" ".join(sentences[i] for i in current_indices))
            current_indices = [k]

    # Añadir el último chunk pendiente al finalizar el recorrido
    if current_indices:
        chunks.append(" ".join(sentences[i] for i in current_indices))

    return chunks


In [32]:
# =============================================================
# SECCIÓN 4 — Pipeline Principal de Ejecución
# =============================================================


def main():
    """
    Orquesta el pipeline completo de preprocesamiento y chunking:

    A. Carga de documentos desde el directorio configurado.
    B. Limpieza y normalización del texto extraído.
    C. Chunking semántico con el algoritmo Max-Min.
    D. Estadísticas de distribución de tokens y exportación a CSV.

    Returns
    -------
    tuple[pd.DataFrame | None, pd.DataFrame | None]
        (df_docs, df_maxmin). Devuelve (None, None) si no hay documentos.
    """
    # ── A. Carga de documentos ────────────────────────────────────────────────
    print(">>> [1/4] Cargando documentos...")
    df_docs = load_documents(RAW_DATA_PATH)

    if df_docs.empty:
        print(f"⚠  No se encontraron documentos en: {RAW_DATA_PATH}")
        print("   Sube archivos PDF/TXT/MD al directorio indicado y vuelve a ejecutar.")
        return None, None

    print(f"    {len(df_docs)} documento(s) cargado(s).")

   # ── B. Limpieza y normalización ───────────────────────────────────────────
    print(">>> [2/4] Limpiando y normalizando texto...")

    reducer1 = NoiseReducer()
    reducer1.change_pattern(1)
    df_docs["clean_text_p1"] = (
        df_docs["text"]
        .apply(reducer1.clean)
        .apply(normalize_text)
    )

    reducer1 = NoiseReducer()
    reducer1.change_pattern(1)
    print(f"Reducer1 tiene {len(reducer1.patterns)} patrones")

    reducer2 = NoiseReducer()
    reducer2.change_pattern(2)
    df_docs["clean_text_p2"] = (
        df_docs["text"]
        .apply(reducer2.clean)
        .apply(normalize_text)
    )
    reducer2 = NoiseReducer()
    reducer2.change_pattern(2)
    print(f"Reducer2 tiene {len(reducer2.patterns)} patrones")

    # ── C. Chunking semántico Max-Min ─────────────────────────────────────────
    print(">>> [3/4] Ejecutando Max-Min Semantic Chunking...")

    df_docs["chunks_p1"] = df_docs["clean_text_p1"].apply(
        lambda text: max_min_chunking(text, hard_thr=0.6, c=0.9, init_const=1.5)
    )
    df_docs["chunks_p2"] = df_docs["clean_text_p2"].apply(
        lambda text: max_min_chunking(text, hard_thr=0.6, c=0.9, init_const=1.5)
    )

    # ── D. Estadísticas y exportación ─────────────────────────────────────────
    print(">>> [4/4] Generando estadísticas y exportando resultados...")

    df_maxmin_p1 = (
        df_docs.explode("chunks_p1")[["file", "chunks_p1"]]
        .rename(columns={"chunks_p1": "chunk_content"})
        .assign(method="Max-Min-P1")
        .reset_index(drop=True)
    )
    df_maxmin_p1["token_count"] = df_maxmin_p1["chunk_content"].apply(
        lambda x: len(encoder.encode(str(x))) if pd.notna(x) else 0
    )

    df_maxmin_p2 = (
        df_docs.explode("chunks_p2")[["file", "chunks_p2"]]
        .rename(columns={"chunks_p2": "chunk_content"})
        .assign(method="Max-Min-P2")
        .reset_index(drop=True)
    )
    df_maxmin_p2["token_count"] = df_maxmin_p2["chunk_content"].apply(
        lambda x: len(encoder.encode(str(x))) if pd.notna(x) else 0
    )

    # Comparar estadísticas antes de continuar
    print("\n── Estadísticas P1 ──")
    print(df_maxmin_p1["token_count"].describe().to_frame().T.to_string(index=False))
    print("\n── Estadísticas P2 ──")
    print(df_maxmin_p2["token_count"].describe().to_frame().T.to_string(index=False))

    df_maxmin_p1.to_csv("resultados_minmax_p1.csv", index=False, encoding="utf-8-sig")
    df_maxmin_p2.to_csv("resultados_minmax_p2.csv", index=False, encoding="utf-8-sig")
    print("\n✔  Corpus exportado: 'resultados_minmax_p1.csv' y 'resultados_minmax_p2.csv'")

    return df_docs, df_maxmin_p1, df_maxmin_p2


# ── Punto de entrada ──────────────────────────────────────────────────────────
df_docs, df_maxmin_p1, df_maxmin_p2 = main()


>>> [1/4] Cargando documentos...
    4 documento(s) cargado(s).
>>> [2/4] Limpiando y normalizando texto...
✅ Usando patterns1 (10 patrones activos)
✅ Usando patterns1 (10 patrones activos)
Reducer1 tiene 10 patrones
✅ Usando patterns2 (55 patrones activos)
✅ Usando patterns2 (55 patrones activos)
Reducer2 tiene 55 patrones
>>> [3/4] Ejecutando Max-Min Semantic Chunking...
>>> [4/4] Generando estadísticas y exportando resultados...

── Estadísticas P1 ──
 count      mean        std  min  25%  50%   75%    max
 417.0 84.235012 101.273917  2.0 34.0 58.0 106.0 1301.0

── Estadísticas P2 ──
 count      mean        std  min   25%  50%   75%    max
 378.0 92.060847 110.865986  3.0 30.25 69.0 112.0 1299.0

✔  Corpus exportado: 'resultados_minmax_p1.csv' y 'resultados_minmax_p2.csv'


In [34]:
# Ver comparación en el primer documento
doc_idx = 0

print("── PATRÓN 1 ──")
print(df_docs["clean_text_p1"].iloc[doc_idx][:500])

print("\n── PATRÓN 2 ──")
print(df_docs["clean_text_p2"].iloc[doc_idx][:500])

── PATRÓN 1 ──
Código   ND-TIFS-ETM-001 

Política de asignación, uso y venta de telefonía  móvil 

01/10/2024 

01/10/2024 

La información contenida en este documento es INTERNA, para uso exclusivo del personal de Spin.  Ninguna de sus partes puede ser circulada, citada o reproducida para distribución fuera de Spin sin  autorización previa y por escrito. 

Interna 

Política de asignación, uso y venta de telefonía móvil  Generales  Descripción  Declarar la política para la asignación, uso y venta de  telefon

── PATRÓN 2 ──
Spin  HOJA  1 de 10 

Dirección Administración y Finzas  

Política de asigción, uso y venta de telefonía  móvil 

01/10/2024 

Fecha de modificación  01/10/2024 

La información contenida en este documento es INTER,  Ningu de sus partes puede ser circulada, citada o reproducida para distribución fuera de Spin sin  autorización previa y por escrito. 

Inter  Dirección Administración y Finzas  Política de asigción, uso y venta de telefonía móvil  Generales  Descrip

In [35]:
# =============================================================
# SECCIÓN 5 — Configuración del Cliente Gemini
# Requisito: API Key registrada en Secretos de Colab como "API_KEY"
# =============================================================

from google import genai
from google.colab import userdata

# Obtener la clave API desde los Secretos de Colab
try:
    API_KEY = userdata.get("API_KEY")
    client = genai.Client(api_key=API_KEY)
    print("✔  Cliente Gemini configurado correctamente.")
except Exception as exc:
    print(f"✖  Error al configurar Gemini: {exc}")
    print("   Verifica que tu API_KEY esté registrada en Secretos de Colab.")
    client = None


✔  Cliente Gemini configurado correctamente.


In [36]:
# =============================================================
# SECCIÓN 6 — Generación del Golden Dataset con LLM (Gemini)
# =============================================================

from google.genai.errors import ServerError

# ── Constantes de configuración ───────────────────────────────────────────────
GEMINI_MODEL = "models/gemini-2.0-flash"   # Modelo Gemini para generación
SAMPLE_SIZE = 15                            # Número de preguntas a generar
MIN_CHUNK_TOKENS = 50                       # Mínimo de tokens por chunk válido
MAX_CHUNK_TOKENS = 800                      # Máximo de tokens por chunk válido
API_SLEEP_SECONDS = 1.5                     # Pausa entre llamadas a la API


def generate_question_with_gemini(
    text_chunk: str,
    gemini_client,
    model: str = GEMINI_MODEL,
) -> str | None:
    """
    Genera una pregunta desafiante sobre un fragmento de texto usando Gemini.

    La pregunta es específica, sin referencias al contexto ("Según el texto...")
    y su respuesta está explícitamente contenida en el fragmento.

    Parameters
    ----------
    text_chunk : str
        Fragmento de texto fuente.
    gemini_client : genai.Client
        Cliente Gemini ya autenticado.
    model : str
        Identificador del modelo Gemini a usar.

    Returns
    -------
    str | None
        Pregunta generada, o None si ocurrió un error.
    """
    prompt = f"""
    Eres un experto en generar datos de prueba para sistemas RAG.
    Tu tarea es leer el siguiente fragmento de texto (contexto) y generar UNA sola pregunta.

    Reglas:
    1. La pregunta debe ser compleja y específica.
    2. La respuesta debe estar EXPLÍCITAMENTE en el texto proporcionado.
    3. No uses frases como "Según el texto" o "En este fragmento".
    4. La pregunta debe tener sentido por sí sola, sin contexto adicional.

    CONTEXTO:
    "{text_chunk}"

    PREGUNTA GENERADA:
    """
    try:
        response = gemini_client.models.generate_content(
            model=model,
            contents=prompt,
        )
        return response.text.strip()
    except ServerError as exc:
        print(f"  ✖ Error de servidor Gemini: {exc}")
        return None
    except Exception as exc:
        print(f"  ✖ Error inesperado: {exc}")
        return None


def create_golden_dataset_llm(
    df_chunks: pd.DataFrame,
    gemini_client,
    sample_size: int = SAMPLE_SIZE,
) -> pd.DataFrame:
    """
    Construye un Golden Dataset de preguntas generadas por LLM.

    Selecciona aleatoriamente chunks válidos (dentro del rango de tokens
    configurado) y genera una pregunta por cada uno con Gemini.

    Parameters
    ----------
    df_chunks : pd.DataFrame
        DataFrame con columnas: chunk_content, token_count, file.
    gemini_client : genai.Client
        Cliente Gemini ya autenticado.
    sample_size : int
        Número máximo de preguntas a generar.

    Returns
    -------
    pd.DataFrame
        Golden Dataset con columnas: question_id, question,
        ground_truth_chunk_id, ground_truth_text, source_file.
    """
    print(f">>> Generando Golden Dataset ({sample_size} muestras) con Gemini...")

    # Filtrar chunks dentro del rango de tokens aceptable
    valid_chunks = df_chunks[
        (df_chunks["token_count"] > MIN_CHUNK_TOKENS)
        & (df_chunks["token_count"] < MAX_CHUNK_TOKENS)
    ].copy()

    if valid_chunks.empty:
        print("⚠  No hay chunks válidos en el rango de tokens configurado.")
        return pd.DataFrame()

    # Submuestrear si hay más chunks disponibles que los necesarios
    if len(valid_chunks) > sample_size:
        valid_chunks = valid_chunks.sample(n=sample_size, random_state=42)

    dataset = []
    total = len(valid_chunks)

    for i, (idx, row) in enumerate(valid_chunks.iterrows(), start=1):
        print(f"  Generando pregunta {i}/{total} (chunk #{idx})...", end=" ")

        question = generate_question_with_gemini(row["chunk_content"], gemini_client)
        time.sleep(API_SLEEP_SECONDS)  # Respetar límites de la API

        if question:
            dataset.append({
                "question_id": str(uuid.uuid4()),
                "question": question,
                "ground_truth_chunk_id": idx,
                "ground_truth_text": row["chunk_content"],
                "source_file": row["file"],
            })
            print("✔")
        else:
            print("✖  (omitido por error)")

    golden_df = pd.DataFrame(dataset)
    print(f"\n✔  Golden Dataset generado: {len(golden_df)} preguntas.")
    return golden_df


# ── Preparar corpus y ejecutar generación ─────────────────────────────────────
for df_maxmin, nombre in [(df_maxmin_p1, "p1"), (df_maxmin_p2, "p2")]:

    df_maxmin = df_maxmin.reset_index(drop=True)
    df_maxmin["token_count"] = df_maxmin["chunk_content"].apply(
        lambda x: len(encoder.encode(str(x)))
    )

    if client is not None:
        print(f"\n>>> Generando Golden Dataset para patrón {nombre}...")
        golden = create_golden_dataset_llm(df_maxmin, client, sample_size=SAMPLE_SIZE)

        if not golden.empty:
            print(f"\nVista previa ({nombre}):")
            print(
                golden[["question", "ground_truth_text", "source_file"]]
                .head()
                .to_string(index=False)
            )
            output = f"golden_dataset_maxmin_{nombre}.csv"
            golden.to_csv(output, index=False, encoding="utf-8-sig")
            print(f"\n✔  Exportado: '{output}'")

            # Guardar en variable nombrada para acceso posterior
            if nombre == "p1":
                golden_p1 = golden
            else:
                golden_p2 = golden
    else:
        print(f"⚠  Saltando Golden Dataset {nombre}: cliente Gemini no disponible.")



>>> Generando Golden Dataset para patrón p1...
>>> Generando Golden Dataset (15 muestras) con Gemini...
  Generando pregunta 1/15 (chunk #240)... ✔
  Generando pregunta 2/15 (chunk #272)... ✔
  Generando pregunta 3/15 (chunk #178)... ✔
  Generando pregunta 4/15 (chunk #350)... ✔
  Generando pregunta 5/15 (chunk #29)... ✔
  Generando pregunta 6/15 (chunk #215)... ✔
  Generando pregunta 7/15 (chunk #331)... ✔
  Generando pregunta 8/15 (chunk #398)... ✔
  Generando pregunta 9/15 (chunk #151)... ✔
  Generando pregunta 10/15 (chunk #256)... ✔
  Generando pregunta 11/15 (chunk #188)... ✔
  Generando pregunta 12/15 (chunk #59)... ✔
  Generando pregunta 13/15 (chunk #365)... ✔
  Generando pregunta 14/15 (chunk #19)... ✔
  Generando pregunta 15/15 (chunk #135)... ✔

✔  Golden Dataset generado: 15 preguntas.

Vista previa (p1):
                                                                                                                                                                     ques

In [41]:
# =============================================================
# SECCIÓN 7 — Motor de Evaluación y Reporte de Métricas RAG
# =============================================================

# Número de resultados a recuperar por consulta
TOP_K = 5


def retrieve_top_k(
    question: str,
    corpus_embeddings,
    df_corpus: pd.DataFrame,
    k: int = TOP_K,
) -> List[dict]:
    """
    Recupera los k chunks más similares a la pregunta por similitud coseno.

    Parameters
    ----------
    question : str
        Pregunta en lenguaje natural.
    corpus_embeddings : torch.Tensor
        Embeddings precalculados de todo el corpus.
    df_corpus : pd.DataFrame
        Corpus de chunks (debe tener columnas: chunk_content, file).
    k : int
        Número de resultados a recuperar.

    Returns
    -------
    List[dict]
        Lista de dicts con: retrieved_chunk_id, score,
        retrieved_text, retrieved_file.
    """
    # Codificar la pregunta en el mismo espacio de embeddings del corpus
    query_embedding = embedding_model.encode(question, convert_to_tensor=True)

    # semantic_search devuelve una lista de listas (una sublista por query)
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=k)[0]

    results = []
    for hit in hits:
        corpus_id = hit["corpus_id"]
        row = df_corpus.iloc[corpus_id]
        results.append({
            "retrieved_chunk_id": corpus_id,
            "score": hit["score"],
            "retrieved_text": row["chunk_content"],
            "retrieved_file": row["file"],
        })
    return results


def calculate_metrics(
    golden_df: pd.DataFrame,
    corpus_df: pd.DataFrame,
    method_name: str,
    k: int = TOP_K,
) -> dict:
    """
    Calcula las métricas de recuperación para un método de chunking.

    Métricas calculadas
    -------------------
    - **Hit Rate**: proporción de consultas donde el chunk exacto
      aparece entre los Top-k resultados.
    - **MRR** (Mean Reciprocal Rank): promedio del recíproco del rango
      del primer resultado relevante.
    - **SRA** (Same Relevant Article): proporción de consultas donde
      el resultado Top-1 pertenece al mismo documento fuente.
    - **RCC** (Relevant Chunk Contained): proporción de consultas donde
      el texto de referencia está contenido en el Top-1 recuperado.

    Parameters
    ----------
    golden_df : pd.DataFrame
        Golden Dataset con columnas: question, ground_truth_chunk_id,
        ground_truth_text, source_file.
    corpus_df : pd.DataFrame
        Corpus de chunks a evaluar.
    method_name : str
        Nombre del método (para el reporte).
    k : int
        Número de resultados a recuperar por consulta.

    Returns
    -------
    dict
        Diccionario con las cuatro métricas promediadas.
    """
    print(f"\n>>> Evaluando '{method_name}' (Top-{k})...")

    # Pre-calcular embeddings del corpus una sola vez (operación costosa)
    print("    Calculando embeddings del corpus...")
    corpus_embeddings = embedding_model.encode(
        corpus_df["chunk_content"].tolist(),
        convert_to_tensor=True,
        show_progress_bar=True,
    )

    # Acumuladores de métricas (se promedian al final)
    acc = {"hit_rate": 0, "mrr": 0.0, "sra": 0, "rcc": 0}
    total_queries = len(golden_df)

    for _, item in golden_df.iterrows():
        results = retrieve_top_k(
            item["question"], corpus_embeddings, corpus_df, k=k
        )

        # ── 1. Hit Rate y MRR: chunk exacto en Top-k ─────────────────────────
        hit = False
        reciprocal_rank = 0.0

        for rank, res in enumerate(results):
            if res["retrieved_chunk_id"] == item["ground_truth_chunk_id"]:
                hit = True
                reciprocal_rank = 1 / (rank + 1)
                break  # Solo se considera el primer acierto

        if hit:
            acc["hit_rate"] += 1
            acc["mrr"] += reciprocal_rank

        # ── 2. SRA: el resultado Top-1 pertenece al mismo documento fuente ────
        top_result = results[0]
        if top_result["retrieved_file"] == item["source_file"]:
            acc["sra"] += 1

        # ── 3. RCC: el texto de referencia está contenido en el Top-1 ─────────
        # Útil cuando el chunk recuperado es más amplio que el de referencia
        # (p. ej., en enfoques de Growing Window).
        gt_text = item["ground_truth_text"].strip()
        retrieved_text = top_result["retrieved_text"].strip()

        if gt_text in retrieved_text:
            acc["rcc"] += 1

    # Promediar cada acumulador sobre el total de consultas
    return {metric: value / total_queries for metric, value in acc.items()}


# ── Ejecutar evaluación y mostrar reporte ─────────────────────────────────────

resultados = {}

for golden, corpus, nombre in [
    (golden_p1, df_maxmin_p1, "Max-Min P1"),
    (golden_p2, df_maxmin_p2, "Max-Min P2"),
]:
    if not golden.empty:
        resultados[nombre] = calculate_metrics(golden, corpus, nombre, k=TOP_K)
    else:
        print(f"⚠  No hay datos en el Golden Dataset {nombre}. Ejecuta la Sección 6 primero.")

if resultados:
    results_df = pd.DataFrame(resultados).T  # filas = métodos, columnas = métricas

    print("\n" + "=" * 52)
    print("      REPORTE DE MÉTRICAS — SPIN COMPASS RAG      ")
    print("=" * 52)
    print(results_df.to_string())
    print("=" * 52)

    # Mejor método según RCC
    best_method = results_df["rcc"].idxmax()
    m = results_df.loc[best_method]

    print(f"\n✔  Mejor método según RCC : {best_method}")
    print(f"   Hit Rate  : {m['hit_rate']:.2%}")
    print(f"   MRR       : {m['mrr']:.4f}")
    print(f"   SRA       : {m['sra']:.2%}")
    print(f"   RCC       : {m['rcc']:.2%}")

else:
    print("⚠  No hay datos en el Golden Dataset. Ejecuta la Sección 6 primero.")



>>> Evaluando 'Max-Min P1' (Top-5)...
    Calculando embeddings del corpus...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]


>>> Evaluando 'Max-Min P2' (Top-5)...
    Calculando embeddings del corpus...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]


      REPORTE DE MÉTRICAS — SPIN COMPASS RAG      
            hit_rate       mrr       sra       rcc
Max-Min P1  0.866667  0.733333  0.733333  0.600000
Max-Min P2  0.533333  0.402222  0.933333  0.333333

✔  Mejor método según RCC : Max-Min P1
   Hit Rate  : 86.67%
   MRR       : 0.7333
   SRA       : 73.33%
   RCC       : 60.00%
